# **RAG : Un LLM répond-il mieux avec une recherche de passages ?**

Ce projet compare quatre méthodes de recherche de passages (BM25, recherche dense, hybride, re-ranking) en mesurant leur effet sur les réponses d'un LLM à 300 questions factuelles, avec des tests statistiques pour distinguer les vrais écarts du hasard.

---
# **Partie 1 : La recherche des meilleurs passages**

1. Les données : des passages Wikipedia et des questions avec leur réponse
2. Nettoyer le texte et noter une réponse
3. Quatre façons de chercher les passages : BM25, recherche dense, hybride, re-ranking

## 0. Installation et imports

In [1]:
!pip install -q sentence-transformers datasets

In [2]:
import re
import string
import time
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", DEVICE)

Device : cuda


## 1. Les données

[`rag-mini-wikipedia`](https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia) contient :
- un **corpus** de 3 200 passages Wikipedia, dans lesquels on va chercher
- 918 **questions**, chacune avec sa réponse attendue.

In [46]:
corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer", split="test")

passages = [p.strip() for p in corpus["passage"] if p.strip()]
questions = qa["question"]
# Une liste par question, car une question peut avoir plusieurs bonnes réponses
answers = [[str(a)] for a in qa["answer"]]

print(len(passages), "passages |", len(questions), "questions")
pd.DataFrame({"question": questions[:5], "réponse": qa["answer"][:5]})

3200 passages | 918 questions


,question,réponse
0,Was Abraham Lincoln the sixteenth President of...,yes
1,Did Lincoln sign the National Banking Act of 1...,yes
2,Did his mother die of pneumonia?,no
3,How many long was Lincoln's formal education?,18 months
4,When did Lincoln begin his political career?,1832


## 2. Nettoyer le texte et noter une réponse

- `tokenize` découpe un texte en mots et retire les mots vides (*the, of, what…*). Elle sert à BM25.
- `normalize_answer` met une réponse sous forme standard pour la comparer (normalisation SQuAD).
- `exact_match`, `token_f1` et `contains_answer` notent une réponse de 0 à 1.

In [5]:
STOPWORDS = set('''
a an and are as at be been but by can did do does for from had has have he her his how i if in into
is it its of on or she so that the their them then there these they this to was we were what when
where which who whom why will with would you your
'''.split())


def strip_accents(text):
    return "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))


def tokenize(text):
    words = re.findall(r"\w+", strip_accents(text.lower()))
    return [w for w in words if w not in STOPWORDS]


def normalize_answer(text):
    text = strip_accents(str(text).lower())                                   # minuscules, sans accents
    text = "".join(" " if ch in string.punctuation else ch for ch in text)    # ponctuation -> espace
    text = re.sub(r"\b(a|an|the)\b", " ", text)                               # retire a / an / the
    return " ".join(text.split())                                             # espaces multiples -> un seul

In [6]:
def exact_match(prediction, golds):
    # 1.0 si la prédiction est égale à au moins une des bonnes réponses
    pred = normalize_answer(prediction)
    return float(any(pred == normalize_answer(g) for g in golds))


def token_f1(prediction, golds):
    # F1 sur les mots : crédit partiel si une partie des mots est juste
    pred_words = normalize_answer(prediction).split()
    best = 0.0
    for gold in golds:
        gold_words = normalize_answer(gold).split()
        common = sum((Counter(pred_words) & Counter(gold_words)).values())
        if common == 0:
            continue
        precision = common / len(pred_words)
        recall = common / len(gold_words)
        best = max(best, 2 * precision * recall / (precision + recall))
    return best


def contains_answer(text, golds):
    # 1.0 si une bonne réponse apparaît comme mot entier dans le texte
    haystack = f" {normalize_answer(text)} "
    return float(any(f" {normalize_answer(g)} " in haystack for g in golds if normalize_answer(g)))


def is_yes_no(golds):
    return all(normalize_answer(g) in {"yes", "no"} for g in golds)


# Test de vérification
print(exact_match("The Impalas.", ["the impalas"]))                    # 1.0
print(token_f1("Michael Moriarty played him", ["Michael Moriarty"]))   # 2* (0.5*1)/(0.5+1) = 0.6667
print(contains_answer("the bus is late", ["us"]))                      # 0.0

1.0
0.6666666666666666
0.0


## 3. Les quatre méthodes de recherche

Elles ont toutes la même forme : `search_xxx(question, k)` renvoie la liste des **numéros** des k meilleurs passages, du plus pertinent au moins pertinent.

On les essaie sur une question d'exemple.

In [10]:
def top_k(scores, k):
    # numéros des k plus grands scores, du plus grand au plus petit
    return list(np.argsort(-scores)[:k])


def show(ids, n_chars=140):
    for rank, i in enumerate(ids, start=1):
        print(f"  {rank}. [{i}] {passages[i][:n_chars]}...")


EXAMPLE = 3
print("Question :", questions[EXAMPLE])
print("Réponse  :", answers[EXAMPLE][0])

Question : How many long was Lincoln's formal education?
Réponse  : 18 months


### 3.1 BM25 : chercher par mots-clés

Score d'un passage = somme, sur les mots de la question, de **IDF × terme saturé** :

$$\text{score}(q, d) = \sum_{\text{mot} \in q} \text{IDF}(\text{mot}) \cdot \frac{tf \cdot (k_1 + 1)}{tf + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{longueur moyenne}}\right)}$$

- **IDF** : un mot rare pèse plus lourd.
- **tf et k₁** : chaque répétition d'un mot rapporte un peu moins que la précédente.
- **b** : les passages longs sont pénalisés.


**Construction (une seule fois, avant toute question).**  On crée  une matrice de taille 3 200 × 20 255 : une ligne équivaut à un passage, une colonne par mot du vocabulaire (l'ensemble des mots *différents* présents dans les passages).  
La case (passage, mot) contient le poids BM25 de ce mot dans ce passage, et 0 si le passage ne contient pas le mot.  
Comme un passage n'utilise qu'une centaine de mots, cette matrice est presque entièrement remplie de zéros. On ne la stocke donc pas en entier : pour chaque mot, on garde seulement la liste des passages qui le contiennent, avec le poids correspondant.



 **Recherche (pour chaque question).** On découpe la question en mots. Pour chaque passage, on additionne les poids des mots de la question qu'il contient : c'est son score BM25. On trie ensuite les passages par score décroissant et on garde les k premiers (fonction top_k).

In [14]:
K1, B = 1.5, 0.75

docs = [tokenize(p) for p in passages]                 # chaque passage découpé en mots
n_docs = len(docs)
doc_len = np.array([len(d) for d in docs])
avg_len = doc_len.mean()

# df : dans combien de passages apparaît chaque mot ?
df = Counter(word for d in docs for word in set(d))
# IDF : plus un mot est rare, plus il pèse
idf = {word: np.log(1 + (n_docs - n + 0.5) / (n + 0.5)) for word, n in df.items()}

# Index inversé : mot -> liste de (numéro du passage, poids BM25 du mot dans ce passage)
bm25_index = defaultdict(list)
for i, d in enumerate(docs):
    for word, tf in Counter(d).items():
        length_norm = 1 - B + B * doc_len[i] / avg_len
        weight = idf[word] * tf * (K1 + 1) / (tf + K1 * length_norm)
        bm25_index[word].append((i, weight))

print(f"Vocabulaire : {len(bm25_index):,} mots différents")

print("Question :", questions[EXAMPLE])    # rappel de la question

def search_bm25(question, k=5):
    scores = np.zeros(n_docs)
    for word in tokenize(question):
        for i, weight in bm25_index.get(word, []):    # seuls les passages qui contiennent le mot
            scores[i] += weight
    return top_k(scores, k)


show(search_bm25(questions[EXAMPLE])) # passages les plus probables de contenir une réponse juste

Vocabulaire : 20,255 mots différents
Question : How many long was Lincoln's formal education?
  1. [376] There were few people who strongly or directly influenced Lincolnâs moral and intellectual development and perspectives. There was no teac...
  2. [287] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, o...
  3. [385] Lincoln has been memorialized in many city names, notably the capital of Nebraska. Lincoln, Illinois, is the only city to be named for Abrah...
  4. [355] Lincoln's powerful rhetoric defined the issues of the war for the nation, the world, and posterity. His extraordinary command of the English...
  5. [387] Abraham Lincoln's birthday, February 12, was formerly a national holiday, now commemorated as Presidents' Day. However, it is still observed...


### 3.2 Recherche dense : chercher par le sens

Chaque texte devient un vecteur de 384 nombres. Des textes de sens proche donnent des vecteurs proches.

- `normalize_embeddings=True` ramène chaque vecteur à une longueur de 1. Le produit scalaire de deux vecteurs est alors leur **cosinus**.
- On encode les 3 200 passages **une seule fois**. Pour chaque question, il suffit d'encoder la question et de faire un produit matrice-vecteur.

In [16]:
encoder = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1", device=DEVICE)

passage_vectors = encoder.encode(passages, normalize_embeddings=True, show_progress_bar=True)
print("Forme de la matrice des passages :", passage_vectors.shape)   # (3200, 384)


def search_dense(question, k=5):
    q = encoder.encode(question, normalize_embeddings=True)   # vecteur de 384 nombres
    scores = passage_vectors @ q                               # 3 200 cosinus d'un coup
    return top_k(scores, k)


show(search_dense(questions[EXAMPLE]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Forme de la matrice des passages : (3200, 384)
  1. [287] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, o...
  2. [292] In 1834, he won election to the state legislature, and after coming across the Commentaries on the Laws of England, began to teach himself l...
  3. [285] Lincoln was just seven years old when, in 1816, the family was forced to make a new start in Perry County (now in Spencer County), Indiana. ...
  4. [288] Young Abraham Lincoln...
  5. [367] An army surgeon, Doctor Charles Leale, initially assessed Lincoln's wound as mortal. The President was taken across the street from the thea...


### 3.3 Hybride : combiner les classements (Reciprocal Rank Fusion)

Les scores de BM25 et les cosinus ne sont pas sur la même échelle, donc on combine les **rangs**. Chaque méthode donne `1 / (60 + rang)` points à chaque passage de son classement, et on additionne.

In [19]:
def search_hybrid(question, k=20, depth=100, c=60):       # on compare les 100 meilleurs de chaque méthode
    points = defaultdict(float)
    for ranking in (search_bm25(question, depth), search_dense(question, depth)):
        for rank, i in enumerate(ranking, start=1):
            points[i] += 1 / (c + rank)
    best = sorted(points, key=points.get, reverse=True)       # du plus de points au moins de points
    return best[:k]


show(search_hybrid(questions[EXAMPLE]))

  1. [287] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, o...
  2. [376] There were few people who strongly or directly influenced Lincolnâs moral and intellectual development and perspectives. There was no teac...
  3. [294] Lincoln wrote a series of anonymous letters, published in 1842 in the Sangamon Journal, mocking State Auditor and prominent Democrat James S...
  4. [390] * American School, Lincoln's economic views....
  5. [306] Lincoln was involved in more than 5,100 cases in Illinois alone during his 23-year legal career. Though many of these cases involved little ...
  6. [367] An army surgeon, Doctor Charles Leale, initially assessed Lincoln's wound as mortal. The President was taken across the street from the thea...
  7. [297] The other Lincoln children were born in Springfield, Illinois, and died either during childhood or their teen years. Edward Baker Lincoln wa...
  8. [1909]

### 3.4 Re-ranking : relire les 20 meilleurs candidats

1. L'hybride sélectionne les **20 meilleurs candidats** parmi les 3 200 passages.
2. Le cross-encoder lit chaque paire (question, passage) et lui donne un score.
3. On trie par score et on garde les k meilleurs.

In [21]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE)


def search_rerank(question, k=5, n_candidates=20):
    candidates = search_hybrid(question, n_candidates)
    scores = reranker.predict([(question, passages[i]) for i in candidates])
    order = np.argsort(-scores)                                # du meilleur score au moins bon
    return [candidates[j] for j in order[:k]]


# Les 4 méthodes de recherche, utilisées dans les sections 4.3 et 5
METHODS = {
    "BM25": search_bm25,
    "Dense": search_dense,
    "Hybride": search_hybrid,
    "Hybride + rerank": search_rerank,
}


show(search_rerank(questions[EXAMPLE]))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  1. [287] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, o...
  2. [376] There were few people who strongly or directly influenced Lincolnâs moral and intellectual development and perspectives. There was no teac...
  3. [306] Lincoln was involved in more than 5,100 cases in Illinois alone during his 23-year legal career. Though many of these cases involved little ...
  4. [285] Lincoln was just seven years old when, in 1816, the family was forced to make a new start in Perry County (now in Spencer County), Indiana. ...
  5. [297] The other Lincoln children were born in Springfield, Illinois, and died either during childhood or their teen years. Edward Baker Lincoln wa...


---
# **Partie 2 : Faire répondre le LLM**

## 4. Le LLM et son prompt

On utilise **Qwen2.5-1.5B-Instruct**

- Le **tokenizer** traduit le texte en numéros de tokens, et inversement.
- Le **modèle** prédit le token suivant.

In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,   # float16 : 2 fois moins de mémoire sur GPU
).to(DEVICE)
model.eval()
print("LLM chargé :", LLM_NAME)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM chargé : Qwen/Qwen2.5-1.5B-Instruct


### 4.1 Le prompt

Deux messages :
- **le message système** donne les consignes générales. On exige une réponse **courte**, sinon l'Exact Match pénaliserait des réponses justes mais détaillées ;
- **le message utilisateur** contient les passages numérotés, puis la question.

Sans passages, c'est la version **closed-book** : le modèle répond de mémoire.

In [25]:
SYSTEM_PROMPT = (
    "You are a question answering assistant. Reply with the shortest possible answer: "
    "a name, a date, a number or a short phrase (usually 1 to 5 words), never a full sentence. "
    "For yes/no questions reply only 'yes' or 'no'."
)


def build_prompt(question, passage_ids=None):
    if not passage_ids:                                    # closed-book : la question seule
        return f"Question: {question}\nShort answer:"
    numbered = "\n\n".join(f"[{n}] {passages[i]}" for n, i in enumerate(passage_ids, start=1))
    return (
        "Answer the question using the passages below.\n\n"
        f"Passages:\n{numbered}\n\n"
        f"Question: {question}\nShort answer:"
    )


# Voir exactement ce que reçoit le LLM (avec 2 passages pour que ce soit lisible)
print(build_prompt(questions[EXAMPLE], search_hybrid(questions[EXAMPLE], 2)))

Answer the question using the passages below.

Passages:
[1] Lincoln's formal education consisted of about 18 months of schooling. Largely self-educated, he read every book he could get his hands on, once walking.  just to borrow one  While his favorite book was The Life of George Washington, Lincoln mastered the Bible, Shakespeare, and English and American history, and developed a plain writing style that puzzled audiences more used to grandiose rhetoric. He was also a talented local wrestler and skilled with an ax; some rails he had allegedly split in his youth were exhibited at the 1860 Republican National Convention, as the party celebrated the poor-boy-made-good theme. Lincoln avoided hunting and fishing because he did not like killing animals, even for food. Though he was unusually tall at  , 4 inches and strong, Lincoln spent so much time reading that some neighbors suspected he must be doing it to avoid strenuous manual labor.

[2] There were few people who strongly or directly

### 4.2 La fonction `answer`

1. On construit les deux messages, puis on les met au format attendu par Qwen (`apply_chat_template` ajoute ses balises spéciales).
2. On transforme le texte en tokens.
3. `generate` produit au plus 24 nouveaux tokens. `do_sample=False` : on prend **toujours le token le plus probable**, pour que les résultats soient reproductibles.
4. `generate` renvoie *prompt + réponse* : on garde seulement les nouveaux tokens.
5. On nettoie : on garde la première ligne, on retire « Answer: », les guillemets et le point final.

In [39]:
def postprocess(text):
    # Nettoie la réponse donnée par le LLM
    text = text.strip().split("\n")[0]
    text = re.sub(r"^(short answer|answer)\s*:\s*", "", text, flags=re.IGNORECASE)
    return text.strip().strip("\"'*").rstrip(".").strip()


def answer(question, passage_ids=None):
    # Les deux messages : les consignes, puis les passages et la question
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(question, passage_ids)},
    ]
    # Assemble les messages en un seul texte au format de conversation de Qwen (balises spéciales)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Transforme ce texte en numéros de tokens et les envoie sur le GPU
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    # Génère la réponse token par token (24 au maximum), en prenant toujours le token le plus probable (do_sample=False) donc les résultats sont reproductibles
    output = model.generate(**inputs, max_new_tokens=24, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    # generate() renvoie prompt + réponse : on garde seulement les nouveaux tokens
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    # Numéros de tokens -> texte, puis nettoyage
    return postprocess(tokenizer.decode(new_tokens, skip_special_tokens=True))


### 4.3 Essai des 5 méthodes sur une question

In [48]:
EXAMPLE = 505
q = questions[EXAMPLE]
print("Question         :", q)
print("Réponse attendue :", answers[EXAMPLE][0])
print()
print(f"{'Closed-book':<18}:", answer(q))                        # sans passage : de mémoire
for name, search in METHODS.items():
    print(f"{name:<18}:", answer(q, search(q, 5)))               # avec les 5 passages de la méthode

Question         : What centred in Sierra?
Réponse attendue : The leopard men

Closed-book       : No
BM25              : West African secret society called Leopard men
Dense             : Ghana
Hybride           : Cerro Catedral
Hybride + rerank  : West African secret society called Leopard men


## 5. Quelle est la meilleure méthode ?

Expérience : On va utiliser **les mêmes questions, le même modèle de LLM, le même prompt**. Seul le contexte va changer :
- *Closed-book* : aucun passage
- puis les 5 meilleurs passages de chacune des 4 méthodes de recherche.

Pour chaque réponse, on calcule l'Exact Match (EM), le F1 et `contains` (la bonne réponse apparaît-elle dans la prédiction ?). On fait ensuite la moyenne par méthode.

On garde aussi `context_has_answer` : la bonne réponse était-elle dans les passages donnés ? Ça servira à l'analyse d'erreurs.

On tire **300 questions au hasard**, avec une graine fixe pour pouvoir refaire l'expérience à l'identique.

In [52]:
N_QA = 300
rng = np.random.default_rng(42)                                      # graine fixe : toujours les mêmes questions
qa_ids = sorted(int(i) for i in rng.choice(len(questions), size=N_QA, replace=False))

SYSTEMS = {"Closed-book": None, **METHODS}                           # None = pas de recherche
rows = []
for name, search in SYSTEMS.items():
    start = time.time()
    for i in qa_ids:
        q, golds = questions[i], answers[i]
        ids = search(q, 5) if search else []
        pred = answer(q, ids)
        rows.append({
            "méthode": name,
            "question": q,
            "réponse attendue": golds[0],
            "prédiction": pred,
            "em": exact_match(pred, golds),
            "f1": token_f1(pred, golds),
            "contains": contains_answer(pred, golds),
            # la bonne réponse était-elle dans les passages ? (pas de sens en closed-book ni pour oui/non)
            "context_has_answer": (float(any(contains_answer(passages[j], golds) for j in ids))
                                   if ids and not is_yes_no(golds) else np.nan),
        })
    print(f"{name:<18} terminé en {time.time() - start:.0f} s")

results = pd.DataFrame(rows)
results.to_csv("resultats_qa.csv", index=False)     # sauvegarde, au cas où Colab se déconnecte

Closed-book        terminé en 51 s
BM25               terminé en 92 s
Dense              terminé en 114 s
Hybride            terminé en 120 s
Hybride + rerank   terminé en 119 s


In [53]:
summary = results.groupby("méthode")[["em", "f1", "contains"]].mean().loc[list(SYSTEMS)]
summary.round(3)

,em,f1,contains
méthode,,,
Closed-book,0.320,0.353,0.323
BM25,0.553,0.643,0.570
Dense,0.540,0.617,0.563
Hybride,0.577,0.656,0.603
Hybride + rerank,0.583,0.670,0.607


### 5.1 Les réponses côte à côte

Pour chaque question, la réponse attendue et la prédiction de chaque version.

In [55]:
side_by_side = results.pivot_table(index=["question", "réponse attendue"], columns="méthode",
                                   values="prédiction", aggfunc="first")[list(SYSTEMS)]
side_by_side.head(15)

,méthode,Closed-book,BM25,Dense,Hybride,Hybride + rerank
question,réponse attendue,,,,,
Adams ' opponents were what?,Democratic Republicans,Democrats,Democratic-Republicans,Thomas Jefferson and Aaron Burr,Democratic-Republican,Democratic-Republicans
Are Gray Wolves native to North America?,Yes,Yes,Yes,Yes,Yes,Yes
Are Kangaroos Shy?,Yes,No,Yes,Yes,Yes,Yes
Are ducks in the Arctic Northern Hemisphere migratory?,Some are,Yes,Yes,Yes,Yes,Yes
Are large pythons potential prey for leopards?,yes,Yes,Yes,Yes,Yes,Yes
Are leopards circular?,No,No,No,No,No,No
Are nationals referred to by the plural : Liechtensteiners ?,Yes,No,Yes,Yes,Yes,Yes
Are otters playful animals?,yes,Yes,Yes,Yes,Yes,Yes
Are otters totem animals?,Yes,Yes,Yes,Yes,Yes,Yes


### 5.2 Conclusion (300 questions, k = 5 passages)

| Méthode | EM | F1 | contains |
|---|---|---|---|
| Closed-book | 0,320 | 0,353 | 0,323 |
| BM25 | 0,553 | 0,643 | 0,570 |
| Dense | 0,540 | 0,617 | 0,563 |
| Hybride | 0,577 | 0,656 | 0,603 |
| Hybride + rerank | 0,583 | 0,670 | 0,607 |

**1. La recherche de passages améliore nettement les réponses du LLM.**  
Sans passages, le modèle répond exactement juste à 32 % des questions. Avec les 5 passages de n'importe quelle méthode, il dépasse 54 %, et atteint 58 % avec les méthodes hybrides.  
Avec 300 questions, l'incertitude sur chaque score est d'environ ±5,6 points (intervalle de confiance à 95 %). Un écart de cette taille ne peut donc pas s'expliquer par le hasard.

**2. Les méthodes hybrides semblent les plus performantes, mais les écarts entre méthodes de recherche restent faibles.**  
L'hybride (0,577) et l'hybride avec re-ranking (0,583) devancent BM25 (0,553) et la recherche dense (0,540).  
Cependant, ces écarts restent trop faibles pour confirmer qu'ils ne relèvent pas du hasard de l'échantillon. (Ces différences devront être confirmées par un test apparié (section 6))


**3. Les erreurs du modèle portent sur le fond, pas sur le format.**   
L'écart entre `contains` et l'EM n'est que de 2 à 3 points pour toutes les méthodes : quand le modèle trouve la bonne réponse, il la donne presque toujours au format attendu.  
L'écart entre le F1 et l'EM, d'environ 8 points, correspond à des réponses partiellement justes, comme « West African secret society called Leopard men » pour « The leopard men ».



## 6. Les écarts entre méthodes sont-ils significatifs ?

Les scores de la section 5 sont mesurés sur **un seul échantillon** de 300 questions. Avec 300 autres questions, on aurait obtenu des chiffres un peu différents. Il faut donc vérifier que les écarts entre méthodes ne sont pas dus au hasard de l'échantillon.

**Le bootstrap** estime cette incertitude à partir de notre seul échantillon :
1. On tire 300 questions au hasard parmi nos 300, **avec remise**.
2. On calcule le score moyen sur ce nouvel échantillon.
3. On répète l'opération 2 000 fois.
4. On calcule l'**intervalle de confiance à 95 %**.

**Le bootstrap apparié** sert à comparer deux méthodes A et B. Comme elles ont répondu aux **mêmes** questions, on calcule pour chaque question la différence de score A − B : +1 si seule A a raison, −1 si seule B a raison, 0 si elles sont d'accord. On applique ensuite le bootstrap à la moyenne de ces différences. Si l'intervalle de confiance **ne contient pas 0**, l'écart est **significatif**.

Comparer question par question est plus précis que de comparer deux intervalles séparés : la difficulté propre à chaque question s'annule, et seules comptent les questions où les deux méthodes ne sont pas d'accord.

In [60]:
def bootstrap_ci(values, n_boot=2000, seed=0):
    # Intervalle de confiance à 95 % de la moyenne, par bootstrap
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    # 2 000 échantillons de même taille, tirés avec remise (chaque ligne = un échantillon d'indices)
    idx = rng.integers(0, len(values), size=(n_boot, len(values)))
    means = values[idx].mean(axis=1)                       # la moyenne de chaque échantillon
    low, high = np.percentile(means, [2.5, 97.5])
    return values.mean(), low, high


# Scores de chaque méthode, dans le même ordre de questions (qa_ids) pour toutes les méthodes
scores = {name: {m: results.loc[results["méthode"] == name, m].to_numpy() for m in ["em", "f1"]}
          for name in SYSTEMS}

### 6.1 Intervalle de confiance de chaque méthode

In [61]:
ci_rows = []
for name in SYSTEMS:
    row = {"méthode": name}
    for metric in ["em", "f1"]:
        mean, low, high = bootstrap_ci(scores[name][metric])
        row[metric] = round(mean, 3)
        row[f"IC 95 % {metric}"] = f"[{low:.3f} ; {high:.3f}]"
    ci_rows.append(row)

pd.DataFrame(ci_rows).set_index("méthode")

,em,IC 95 % em,f1,IC 95 % f1
méthode,,,,
Closed-book,0.320,[0.263 ; 0.370],0.353,[0.300 ; 0.402]
BM25,0.553,[0.497 ; 0.607],0.643,[0.593 ; 0.692]
Dense,0.540,[0.480 ; 0.597],0.617,[0.565 ; 0.665]
Hybride,0.577,[0.520 ; 0.630],0.656,[0.609 ; 0.706]
Hybride + rerank,0.583,[0.527 ; 0.637],0.670,[0.622 ; 0.716]


**Résultats**  

L’IC du closed-book ne chevauche aucun IC des méthodes avec recherche. La recherche fait clairement mieux.

Entre les quatre méthodes de recherche, les IC se chevauchent largement. En les regardant séparément, on ne peut pas les départager.

### 6.2 Comparer deux méthodes : le bootstrap apparié (sur l'EM)

Les IC séparés se chevauchent, mais ce n'est pas suffisant pour conclure que les méthodes sont équivalentes. Comme toutes les méthodes ont répondu aux mêmes 300 questions, on peut les comparer question par question.

Pour deux méthodes A et B, on calcule pour chaque question la différence d'EM, A − B :

| Question | EM de A | EM de B | A − B |
|---|---|---|---|
| 1 | 1 | 1 | 0 (toutes les deux justes) |
| 2 | 0 | 0 | 0 (toutes les deux fausses) |
| 3 | 1 | 0 | **+1** (seule A est juste) |
| 4 | 0 | 1 | **−1** (seule B est juste) |

La moyenne de ces différences est l'écart d'EM entre A et B. On applique ensuite le bootstrap à cette colonne de différences : 2 000 tirages avec remise, 2 000 moyennes, puis l'intervalle entre le 2,5ᵉ et le 97,5ᵉ percentile.

**Règle de décision :**
- si l'IC de la différence **ne contient pas 0**, l'écart est **significatif** : une méthode est réellement meilleure ;
- s'il **contient 0**, on ne peut pas conclure : la vraie différence pourrait être nulle.

  

Dans le tableau de résultats, les colonnes **« A seul juste »** et **« B seul juste »** donnent le nombre de ces désaccords : c'est sur eux que repose toute la comparaison.

In [62]:
def compare(a, b, metric="em"):
    sa, sb = scores[a][metric], scores[b][metric]
    mean, low, high = bootstrap_ci(sa - sb)                # différence question par question
    return {
        "comparaison": f"{a} − {b}",
        "Δ EM": round(mean, 3),
        "IC 95 %": f"[{low:+.3f} ; {high:+.3f}]",
        "significatif": "oui" if not (low <= 0 <= high) else "non",
        f"{a} seul juste": int(((sa == 1) & (sb == 0)).sum()),   # questions où seule méthode A a raison
        f"{b} seul juste": int(((sa == 0) & (sb == 1)).sum()),   # questions où seule méthode B a raison
    }


comparisons = pd.DataFrame([
    compare("BM25", "Closed-book"),
    compare("Hybride", "BM25"),
    compare("Hybride", "Dense"),
    compare("BM25", "Dense"),
    compare("Hybride + rerank", "Hybride"),
]).set_index("comparaison")
comparisons

,Δ EM,IC 95 %,significatif,BM25 seul juste,Closed-book seul juste,Hybride seul juste,Dense seul juste,Hybride + rerank seul juste
comparaison,,,,,,,,
BM25 − Closed-book,0.233,[+0.180 ; +0.290],oui,80.0,10.0,NaN,NaN,NaN
Hybride − BM25,0.023,[-0.003 ; +0.050],non,5.0,NaN,12.0,NaN,NaN
Hybride − Dense,0.037,[+0.013 ; +0.063],oui,NaN,NaN,13.0,2.0,NaN
BM25 − Dense,0.013,[-0.020 ; +0.047],non,15.0,NaN,NaN,11.0,NaN
Hybride + rerank − Hybride,0.007,[-0.020 ; +0.033],non,NaN,NaN,7.0,NaN,9.0


**Résultats**

| Comparaison (A − B) | Δ EM | IC 95 % | Significatif | A seul juste | B seul juste |
|---|---|---|---|---|---|
| BM25 − Closed-book | +0,233 | [+0,180 ; +0,290] | oui | 80 | 10 |
| Hybride − BM25 | +0,023 | [−0,003 ; +0,050] | non | 12 | 5 |
| Hybride − Dense | +0,037 | [+0,013 ; +0,063] | oui | 13 | 2 |
| BM25 − Dense | +0,013 | [−0,020 ; +0,047] | non | 15 | 11 |
| Hybride + rerank − Hybride | +0,007 | [−0,020 ; +0,033] | non | 9 | 7 |

**1. La recherche améliore très nettement les réponses du LLM.**
Sur les 90 questions où BM25 et le closed-book ne sont pas d'accord, BM25 donne la bonne réponse 80 fois. Il reste 10 questions où le modèle connaissait la réponse de mémoire, mais où les passages l'ont induit en erreur.

**2. L'hybride est significativement meilleur que la recherche dense.**
Pris séparément, leurs IC se chevauchent. Le test apparié montre pourtant que, sur les 15 questions où elles ne sont pas d'accord, l'hybride a raison 13 fois. Combiner les mots-clés (BM25) et le sens (recherche dense) corrige donc une partie des erreurs de la recherche dense seule.

**3. L'avantage de l'hybride sur BM25 n'est pas démontré.**
L'hybride gagne 12 désaccords contre 5, mais l'IC [−0,003 ; +0,050] contient 0 de justesse. C'est une tendance favorable, qu'il faudrait confirmer sur davantage de questions.

**4. BM25 et la recherche dense sont équivalents** sur ce corpus (15 désaccords gagnés contre 11, écart non significatif).

**5. Le re-ranking n'apporte aucun gain.**
Avec 9 désaccords gagnés contre 7, il est équivalent à l'hybride seul, alors qu'il ajoute un modèle et du temps de calcul. Sur ce corpus, la fusion BM25 + recherche dense suffit.


---
# **Conclusion générale**

## La question posée
Un LLM répond-il mieux à des questions factuelles quand on lui fournit des passages de Wikipedia (RAG), et quelle méthode de recherche de passages est la plus efficace ?

## Ce qui a été fait
- **4 méthodes de recherche** sur un corpus de 3 200 passages : BM25 (mots-clés, implémenté à la main), recherche dense (embeddings), hybride (fusion des deux classements par Reciprocal Rank Fusion) et hybride + re-ranking (cross-encoder).
- **Un même LLM** (Qwen2.5-1.5B-Instruct) répond aux **mêmes 300 questions**, sans passage (*closed-book*) puis avec les 5 passages de chaque méthode.
- **Évaluation** par Exact Match, F1 et `contains`, avec intervalles de confiance par bootstrap et comparaisons appariées.

## Les résultats
1. **La recherche de passages améliore très nettement les réponses.** L'EM passe de **0,32** sans passages à **0,58** avec les méthodes hybrides (+26 points). C'est le résultat le plus solide de l'étude : l'écart est largement significatif.
2. **L'hybride est la méthode la plus fiable.** Il bat significativement la recherche dense (13 désaccords gagnés sur 15) et devance BM25, sans que cet avantage soit encore démontré (IC [−0,003 ; +0,050]).
3. **Le re-ranking n'apporte aucun gain** sur ce corpus, alors qu'il ajoute du temps de calcul.
4. **Les erreurs restantes portent sur le fond, pas sur le format** : l'écart entre `contains` et l'EM n'est que de 2 à 3 points.

**Recommandation :** pour ce type de corpus, la recherche **hybride BM25 + dense, sans re-ranking**, offre le meilleur compromis entre qualité des réponses et coût de calcul.

## Limites
- **Un seul LLM, de petite taille** (1,5 milliard de paramètres). Avec un modèle plus grand, le score du closed-book serait plus élevé et l'apport de la recherche probablement plus faible.
- **Un corpus réduit** (3 200 passages sur des sujets proches). Sur un corpus de plusieurs millions de passages, la recherche serait plus difficile et les écarts entre méthodes sans doute plus marqués.
- **Une seule valeur de k** (5 passages) et des paramètres non optimisés (k₁ et b de BM25, constante de la RRF, 20 candidats pour le re-ranking).
- **300 questions** : les écarts entre méthodes de recherche (2 à 4 points) sont proches du seuil de détection. Cinq tests ont été réalisés sans correction pour les comparaisons multiples : le seul écart significatif entre méthodes de recherche (hybride contre dense) reste à confirmer.
- **L'Exact Match est strict** : une réponse juste mais formulée autrement (« 18 months » contre « a year and a half ») est comptée fausse.

## Pistes d'amélioration
- **Analyse d'erreurs** : quand le LLM se trompe, la bonne réponse était-elle dans les passages ? Cela dirait s'il faut améliorer la recherche ou le modèle. La colonne `context_has_answer` le permet déjà.
- Étudier les **10 questions où les passages ont induit le modèle en erreur**, alors qu'il connaissait la réponse de mémoire.
- Tester **plusieurs valeurs de k** et un **LLM plus grand**.